# 🧮 API Batch Calculator

**Universal utility** for planning any embeddings/LLM batch job against cloud APIs (OpenAI, Azure, Cohere, Anthropic, Voyage, etc.).

Before throwing **31,216 chunks** (or **1,000,000**, or **150**) at an API, this notebook tells you:

- How many **batches** you need
- How long it will **take**
- How much it will **cost**
- Which is the **bottleneck** (TPM or RPM)

**How to use**: edit 2 cells (model config + corpus config) and re-run everything below. It's an interactive calculator without fancy widgets.

---

## Index

1. **Core concepts** (theory with analogies)
2. **Calculator** (configure your model + your corpus)
3. **Results** (table with everything you need to know)
4. **Visualizations** (3 plots to see how it scales)
5. **Comparative presets** (OpenAI, Cohere, Anthropic, BGE, Voyage)

## 1.1 — What is a token?

A **token** is a small piece of text. It's not exactly a word — it can be:

- A whole word: `"Apple"` = 1 token
- Part of a word: `"unbelievable"` = 3 tokens (`"un"`, `"believ"`, `"able"`)
- Spaces and punctuation count too

**Mental rule of thumb**: in English, **1 token ≈ 0.75 words** or **1 word ≈ 1.3 tokens**.

> Example: `"smoke test from Foundry"` = 4 words ≈ **5-6 tokens**.

**Why does it matter?** API prices and limits are measured in **tokens**, not words or characters.

## 1.2 — What is a request?

A **request** = one API call. It's like ringing the doorbell at a pizzeria: *"Hi, I want to place an order"*.

Each request carries:

- **Inputs**: what you send (in embeddings, a list of texts to vectorize)
- **Output**: what you get back (in embeddings, a list of vectors)

> Example: if in one request you send 100 chunks of text, OpenAI returns 100 vectors. That's **1 request with 100 inputs**.

## 1.3 — The 3 typical rate limits

Almost all cloud APIs have these 3 limits. Think of a **pizzeria**:

### Limit 1 — Maximum input size (`max_tokens_per_input`)

> *"I can't cook a pizza bigger than 8,192 g."*

Each individual text CANNOT exceed N tokens. For `text-embedding-3-small` the max is 8,192 tokens. If you send a text with 10,000 tokens, you get an error.

### Limit 2 — How many orders per minute (`RPM` = Requests Per Minute)

> *"I can take a maximum of 6,000 orders per minute, don't call me more often than that."*

RPM = Requests Per Minute. NO more than N API calls in 1 minute. If you exceed → **error 429 (Too Many Requests)**.

### Limit 3 — How much food it processes per minute (`TPM` = Tokens Per Minute) ⭐ THE ONE THAT BITES

> *"My kitchen can only prepare 1,000,000 g of pizza per minute, no matter how many orders you place."*

TPM = Tokens Per Minute. The actual capacity to process text. **Almost always the real bottleneck.**

## 1.4 — The full pizzeria (analogy applied)

Imagine you walk into a pizzeria for a party:

- You need **31,216 mini-pizzas**
- The pizzeria has **3 rules**:
  1. Each individual pizza: max 8,192 g (we comply, ours are ~512 g)
  2. Max 2,048 pizzas per order (we comply by placing ~16 orders)
  3. The kitchen processes max 1,000,000 g/minute (this is the real limit)
- Each of your orders weighs ~1M g (~2,048 pizzas × ~500 g each)
- The kitchen takes **~1 minute per order** (1M g ÷ 1M g/min)
- You place 16 orders sequentially with short pauses ≈ **~30 minutes total**

If you try to order all 31K in a single order, they say *"no, max 2,048 per order, divide it"*. If you call too often, they say *"wait"*. That's why we pause a bit between orders.

## 1.5 — Why is TPM almost always the bottleneck?

If you wanted to go faster, you might think *"let's make smaller batches but send way more requests per minute"*. But the TPM limits you anyway:

| Strategy | RPM used | TPM used | Does it work? |
|----------|----------|----------|---------------|
| 16 batches × 2,048 chunks | 16 RPM (far from 6,000 max) | 1M TPM (full) | ✅ Same speed |
| 312 batches × 100 chunks | 312 RPM (also far) | 1M TPM (full) | ✅ Same speed |
| 31,000 batches × 1 chunk | 31,000 RPM (EXCEEDS the 6,000 max) | 1M TPM | ❌ 429 error |

**Key takeaway**: since **TPM is the real bottleneck**, it almost doesn't matter how many requests you make. The total speed is fixed by the TPM.

**You only gain time if**:
- Microsoft increases your TPM (deployment type upgrade or quota request)
- You use multiple regions in parallel (1M TPM per region × N regions)
- You negotiate a dedicated PTU deployment

---

## The 3 things to remember for ANY future project

1. **Every API has 2-3 limits**: max input size, RPM, TPM. Identify them before coding.
2. **TPM is almost always the bottleneck** in AI APIs. It defines the minimum speed.
3. **Always persist results incrementally** — save after each successful batch. If it crashes, you resume.

---

# 2 — Calculator

Now we understand the concepts. Time for the actual calculator.

**How it works**:

1. Run the imports + helpers cell (next)
2. Edit the `model = ModelConfig(...)` cell with your API limits
3. Edit the `corpus = CorpusConfig(...)` cell with your corpus size
4. Run all cells below to see results + plots
5. Change anything and re-run → updated results instantly

In [ ]:
"""API Batch Calculator — helpers and data structures.

Zero new dependencies: uses only what the repo already has
(matplotlib, pandas, dataclasses).
"""
from dataclasses import dataclass

import matplotlib.pyplot as plt
import pandas as pd


@dataclass
class ModelConfig:
    """Rate limits + pricing for any cloud embedding/LLM model."""
    name: str                            # display name
    tpm: int                             # tokens per minute
    rpm: int                             # requests per minute
    max_inputs_per_request: int          # max array size in one POST
    max_tokens_per_input: int            # max length per individual input
    price_per_1m_tokens_usd: float       # USD per 1M tokens


@dataclass
class CorpusConfig:
    """Your job: how many texts and average size."""
    n_chunks: int                        # number of texts to embed
    avg_tokens_per_chunk: int            # average tokens per chunk


def compute_batch_plan(
    model: ModelConfig,
    corpus: CorpusConfig,
    tpm_safety_margin: float = 0.7,           # use 70% of TPM as safety margin
    pause_seconds_between_batches: float = 60.0,
) -> dict:
    """Compute everything needed to plan a batch run.

    Returns dict with: total_tokens, n_batches, time estimates, cost, bottleneck.
    """
    # 1. Total tokens
    total_tokens = corpus.n_chunks * corpus.avg_tokens_per_chunk

    # 2. Effective TPM with safety margin
    tpm_effective = int(model.tpm * tpm_safety_margin)

    # 3. Max chunks per batch (limited by max_inputs OR max_tokens-per-batch)
    max_tokens_per_batch = int(model.tpm * 0.8)  # batch shouldn't be >80% of TPM/min
    chunks_by_input_limit = model.max_inputs_per_request
    chunks_by_token_limit = max(1, max_tokens_per_batch // corpus.avg_tokens_per_chunk)
    chunks_per_batch = min(chunks_by_input_limit, chunks_by_token_limit)

    # 4. Number of batches (ceiling division)
    n_batches = (corpus.n_chunks + chunks_per_batch - 1) // chunks_per_batch

    # 5. Tokens per batch (most are full, last one may be smaller)
    tokens_per_batch = chunks_per_batch * corpus.avg_tokens_per_chunk

    # 6. Time per batch (TPM-limited)
    embed_time_per_batch_min = tokens_per_batch / tpm_effective

    # 7. Total time including pauses
    total_pause_min = (n_batches - 1) * (pause_seconds_between_batches / 60)
    total_embed_min = n_batches * embed_time_per_batch_min
    total_time_min = total_embed_min + total_pause_min

    # 8. Total cost
    total_cost_usd = total_tokens / 1_000_000 * model.price_per_1m_tokens_usd

    # 9. Bottleneck analysis
    if total_time_min > 0:
        rpm_used = n_batches / total_time_min
        tpm_used = total_tokens / total_time_min
        rpm_pct = rpm_used / model.rpm * 100 if model.rpm > 0 else 0
        tpm_pct = tpm_used / model.tpm * 100 if model.tpm > 0 else 0
    else:
        rpm_pct = tpm_pct = 0
    bottleneck = "TPM" if tpm_pct > rpm_pct else "RPM"

    return {
        "total_tokens": total_tokens,
        "chunks_per_batch": chunks_per_batch,
        "n_batches": n_batches,
        "tokens_per_batch": tokens_per_batch,
        "embed_time_per_batch_min": embed_time_per_batch_min,
        "total_embed_min": total_embed_min,
        "total_pause_min": total_pause_min,
        "total_time_min": total_time_min,
        "total_cost_usd": total_cost_usd,
        "tpm_pct_used": tpm_pct,
        "rpm_pct_used": rpm_pct,
        "bottleneck": bottleneck,
    }

print("✅ Helpers loaded — ModelConfig, CorpusConfig, compute_batch_plan")

## ⚠️ Before configuring — what's fixed and what's variable?

There are 5 parameters in `ModelConfig`. **3 are model constants** (you read them from docs, you don't invent them) and **2 depend on your specific deployment** (you read them from the portal):

| Parameter | Where you get it | Changes if you change deployment? |
|-----------|------------------|-----------------------------------|
| `max_tokens_per_input` | Model docs (fixed) | ❌ No |
| `max_inputs_per_request` | API docs (fixed) | ❌ No |
| `price_per_1m_tokens_usd` | Pricing page (fixed) | ❌ No |
| **`tpm`** | Portal → Deployment Details | ✅ Yes (depends on deployment type) |
| **`rpm`** | Portal → Deployment Details | ✅ Yes (depends on deployment type) |

### Visual tree — model constants vs deployment variables

```
text-embedding-3-small (the model)
├── Model constants (fixed for everyone)
│   ├── max_tokens_per_input = 8,192
│   ├── max_inputs_per_request = 2,048
│   └── price = $0.02/1M tokens
│
└── Your specific deployment (variables by deployment type)
    ├── deployment_type = "Global Standard"
    ├── tpm = 1,000,000   (Global Standard; if Standard would be 350K)
    └── rpm = 6,000        (Global Standard; if Standard would be 2,100)
```

**Mental rule**: if you change the deployment type (Standard → Global Standard → PTU), only `tpm` and `rpm` change. The rest stays the same because it's the model's, not the deployment's.

## 2.1 — Configure your model

👇 **Edit the values below with your API limits**. You can find them in the provider's portal (Azure → Models + endpoints, OpenAI Platform → Limits, Cohere → Account → Quotas, etc.).

In [ ]:
# 👇 EDIT these values with your model limits
model = ModelConfig(
    name="text-embedding-3-small (Azure OpenAI, GlobalStandard)",
    tpm=1_000_000,                     # tokens per minute (from portal)
    rpm=6_000,                          # requests per minute (from portal)
    max_inputs_per_request=2_048,       # API constant — max array size per POST
    max_tokens_per_input=8_192,         # API constant — max length per individual input
    price_per_1m_tokens_usd=0.02,       # USD per 1M tokens (from pricing page)
)
print(f"✅ Model configured: {model.name}")

## 2.2 — Configure your corpus

👇 **Edit with your corpus size**. If you don't know the exact average tokens, a rough estimate works (e.g., 512 if you chunked at 512 tokens).

In [ ]:
# 👇 EDIT with your corpus size
corpus = CorpusConfig(
    n_chunks=31_216,             # number of texts to embed
    avg_tokens_per_chunk=403,    # average tokens per chunk
)
print(f"✅ Corpus configured: {corpus.n_chunks:,} chunks × {corpus.avg_tokens_per_chunk} avg tokens")

---

# 3 — Results

Run the cell below to see the complete batch run plan.

In [ ]:
# Compute the plan
plan = compute_batch_plan(model, corpus)

# Pretty print results
print("=" * 64)
print(f"  BATCH PLAN — {model.name}")
print("=" * 64)
print()
print("📊 Corpus")
print(f"   chunks                     : {corpus.n_chunks:,}")
print(f"   tokens/chunk (avg)         : {corpus.avg_tokens_per_chunk:,}")
print(f"   total tokens               : {plan['total_tokens']:,} (~{plan['total_tokens']/1e6:.2f}M)")
print()
print("📦 Batching")
print(f"   chunks per batch           : {plan['chunks_per_batch']:,}")
print(f"   tokens per batch           : {plan['tokens_per_batch']:,}")
print(f"   number of batches          : {plan['n_batches']}")
print()
print("⏱️  Timing")
print(f"   embed time / batch         : {plan['embed_time_per_batch_min']:.2f} min")
print(f"   total embed time           : {plan['total_embed_min']:.1f} min")
print(f"   total pause time           : {plan['total_pause_min']:.1f} min")
print(f"   ★ TOTAL ESTIMATED TIME     : {plan['total_time_min']:.1f} min ({plan['total_time_min']/60:.2f} h)")
print()
print("💰 Cost")
print(f"   ★ TOTAL COST               : ${plan['total_cost_usd']:.4f}")
print()
print("🎯 Bottleneck analysis")
print(f"   TPM used                   : {plan['tpm_pct_used']:.1f}% of {model.tpm:,}")
print(f"   RPM used                   : {plan['rpm_pct_used']:.1f}% of {model.rpm:,}")
print(f"   ★ Bottleneck is            : {plan['bottleneck']}")

## 📐 Grounded math (no metaphors) — step by step

The compact calculation behind the print above, with our actual numbers:

**Inputs**:
- `n_chunks` = 31,216
- `avg_tokens_per_chunk` = 403
- `tpm` = 1,000,000 · `rpm` = 6,000
- `max_inputs_per_request` = 2,048
- `tpm_safety_margin` = 70%

**Step 1 — Total tokens**

```
31,216 chunks × 403 tokens/chunk = 12,580,048 tokens (~12.58M)
```

**Step 2 — Effective TPM (with safety margin)**

```
1,000,000 × 0.70 = 700,000 tokens/min effective
```

**Step 3 — Chunks per batch (smaller of two limits)**

```
By array limit:  2,048 chunks (max_inputs_per_request)
By token limit:  800,000 tokens / 403 ≈ 1,985 chunks (80% of TPM cap)
chunks_per_batch = min(2,048, 1,985) = 1,985
```

**Step 4 — Number of batches (ceiling division)**

```
ceil(31,216 / 1,985) = 16 batches
```

**Step 5 — Tokens per batch**

```
1,985 × 403 ≈ 800,055 tokens per batch
```

**Step 6 — Embed time per batch (TPM-limited)**

```
800,055 / 700,000 ≈ 1.14 min per batch
```

**Step 7 — Total time**

```
Embed time:  16 × 1.14 ≈ 18.3 min
Pause time:  15 × 60s = 15 min     (15 pauses between 16 batches)
TOTAL    ≈ ~33 min
```

**Step 8 — Total cost**

```
12,580,048 / 1,000,000 × $0.02 = $0.252
```

**Step 9 — Bottleneck check**

```
TPM utilization:  ~100% (1M tokens/min saturated each batch)
RPM utilization:  ~0.5% (16 batches in 33 min = 0.5 RPM, far from 6,000 max)
→ Bottleneck = TPM
```

**Translation**: with this deployment, our 31K chunks job takes **~33 min and costs $0.25**. The TPM is what limits speed; the RPM has tons of headroom.

---

# 4 — Visualizations

3 plots to understand how the batch run scales:

1. **Total time vs # of chunks** → how time grows with corpus size
2. **Bottleneck by batch size** → when TPM vs RPM limits you
3. **Total cost vs # of chunks** → for budgeting

Each plot marks **your current corpus in red** on the curve.

In [ ]:
# Plot 1: Total time vs # of chunks (linear scale)
chunks_range = [1_000, 5_000, 10_000, 20_000, 30_000, 50_000, 100_000, 200_000]
times = []
for n in chunks_range:
    sim_corpus = CorpusConfig(n_chunks=n, avg_tokens_per_chunk=corpus.avg_tokens_per_chunk)
    sim_plan = compute_batch_plan(model, sim_corpus)
    times.append(sim_plan["total_time_min"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(chunks_range, times, marker="o", linewidth=2)
ax.scatter([corpus.n_chunks], [plan["total_time_min"]], color="red", zorder=5, s=100,
           label=f"Your corpus ({corpus.n_chunks:,} chunks → {plan['total_time_min']:.0f} min)")
ax.set_xlabel("Number of chunks")
ax.set_ylabel("Total time (minutes)")
ax.set_title(f"Total time vs corpus size — {model.name}")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Bottleneck by batch size
# Small batches → saturate RPM first; large batches → saturate TPM first.
chunks_per_batch_range = [10, 50, 100, 250, 500, 1_000, 2_000]
tpm_pcts, rpm_pcts = [], []

for cpb in chunks_per_batch_range:
    sim_n_batches = (corpus.n_chunks + cpb - 1) // cpb
    sim_tokens_per_batch = cpb * corpus.avg_tokens_per_chunk
    sim_embed_time_per_batch = sim_tokens_per_batch / (model.tpm * 0.7)
    sim_total_time = sim_n_batches * sim_embed_time_per_batch
    if sim_total_time > 0:
        tpm_used = (sim_tokens_per_batch * sim_n_batches) / sim_total_time
        rpm_used = sim_n_batches / sim_total_time
        tpm_pcts.append(tpm_used / model.tpm * 100)
        rpm_pcts.append(rpm_used / model.rpm * 100 if model.rpm > 0 else 0)
    else:
        tpm_pcts.append(0); rpm_pcts.append(0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(chunks_per_batch_range, tpm_pcts, marker="o", label="TPM utilization (%)", linewidth=2)
ax.plot(chunks_per_batch_range, rpm_pcts, marker="s", label="RPM utilization (%)", linewidth=2)
ax.axhline(100, color="red", linestyle="--", alpha=0.5, label="100% limit")
ax.set_xscale("log")
ax.set_xlabel("Chunks per batch (log scale)")
ax.set_ylabel("% of limit utilized")
ax.set_title("Which limit bottlenecks you? Depends on batch size")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Total cost vs # of chunks (linear)
chunks_range = [1_000, 5_000, 10_000, 20_000, 30_000, 50_000, 100_000, 200_000]
costs = [n * corpus.avg_tokens_per_chunk / 1_000_000 * model.price_per_1m_tokens_usd for n in chunks_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(chunks_range, costs, marker="o", linewidth=2, color="green")
ax.scatter([corpus.n_chunks], [plan["total_cost_usd"]], color="red", zorder=5, s=100,
           label=f"Your corpus (${plan['total_cost_usd']:.4f})")
ax.set_xlabel("Number of chunks")
ax.set_ylabel("Total cost (USD)")
ax.set_title(f"Cost vs corpus size — {model.name} @ ${model.price_per_1m_tokens_usd}/1M tokens")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

---

# 5 — Comparative presets

Table comparing the most popular embedders/LLMs **for the corpus you configured above**.

Quotas are approximate (vary by account tier and region). Edit the `presets` list to add/remove models you want to compare.

In [ ]:
# Popular model presets (approximate quotas, vary by tier/region)
presets = [
    # Embedders
    ModelConfig("OpenAI text-embedding-3-small (direct)", 1_000_000, 5_000, 2_048, 8_192, 0.02),
    ModelConfig("OpenAI text-embedding-3-large (direct)", 1_000_000, 5_000, 2_048, 8_192, 0.13),
    ModelConfig("Azure OpenAI text-embedding-3-small (S0)", 350_000, 2_100, 2_048, 8_192, 0.02),
    ModelConfig("Cohere embed-english-v3.0", 100_000, 10_000, 96, 512, 0.10),
    ModelConfig("Voyage voyage-3-large", 100_000, 2_000, 128, 32_000, 0.06),
    ModelConfig("BGE-M3 self-hosted (MPS, no API limits)", 999_999_999, 999_999_999, 9_999, 8_192, 0.0),
    # For chat (not embeddings, but comparable for batch planning)
    ModelConfig("Anthropic Claude Sonnet 4.6 (input)", 200_000, 4_000, 1, 200_000, 3.00),
]

rows = []
for preset in presets:
    p = compute_batch_plan(preset, corpus)
    rows.append({
        "Model": preset.name,
        "Total tokens": f"{p['total_tokens']:,}",
        "# Batches": p["n_batches"],
        "Total time": f"{p['total_time_min']:.1f} min",
        "Total cost": f"${p['total_cost_usd']:.4f}",
        "Bottleneck": p["bottleneck"],
    })

df = pd.DataFrame(rows)
print(f"Comparing {corpus.n_chunks:,} chunks × {corpus.avg_tokens_per_chunk} avg tokens:\n")
df

---

## ⚠️ Important disclaimer about quotas

Portal limits **are NOT always the effective limits for your account**. The **tier** (free/S0/S1/Standard/Enterprise) determines the actual quota. For example:

- Azure AI Services tier **S0** often has much lower quotas than the portal shows
- OpenAI **Free** tier has restrictive rate limits until you go paid
- Cohere **Trial** keys have a 10 requests/min limit (not 10,000)

**Before any large batch run, always do a short test** (e.g., `--limit 50`) to discover whether effective limits are those of the portal or lower.